# Models

## Initialize a model and set common parameters.

In [ ]:
import os
from pydantic import SecretStr

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"]),
    temperature=0.7,
    max_tokens=1000,
    timeout=3000,
    max_retries=3
)

response = model.invoke("Why do parrots have colorful feathers?")
print(response.text)

**Short answer:**  
Parrots’ bright plumage comes from a mix of pigments and microscopic feather structures that reflect light. These colors serve several evolutionary purposes—most importantly attracting mates, signaling health, helping individuals recognize members of their own species, and (in some habitats) providing camouflage or warning signals.

---

## 1. How the colors are produced  

| Type of coloration | What creates it | Typical colors in parrots |
|-------------------|----------------|---------------------------|
| **Pigment‑based** | **Carotenoids** (diet‑derived) → reds, oranges, yellows <br>**Melanins** (produced internally) → blacks, browns, grays | Red, orange, yellow, brown, black |
| **Structural** | Nanoscale keratin‑air‑filled “spongy” layers in the feather barbs that scatter light → **iridescence** and **blue/green** hues (the actual pigment is often absent; the color is produced by the way light interferes) | Bright blues, greens, turquoise, sometimes violet |


## Stream tokens as they arrive, and batch several prompts at once.

In [6]:
for chunk in model.stream("Explain aeroplanes in one sentence"):
    print(chunk.text, end="", flush=True)

answers = model.batch([
    "Why is the sky blue?",
    "How do airplanes fly?",
])

for ans in answers:
    ans.pretty_print()

An aeroplane is a powered, heavier‑than‑air vehicle whose fixed wings generate lift by moving through the air, allowing it to fly and transport people or cargo.================================== Ai Message ==================================

**Short answer:**  
The sky is blue because molecules and tiny particles in Earth’s atmosphere scatter sunlight. Short‑wavelength light (blue and violet) is scattered far more efficiently than long‑wavelength light (red, orange). Our eyes are more sensitive to blue than violet and the violet light is partly absorbed by the upper atmosphere, so the net color we see is blue.

---

## 1. The physics behind the color

### 1.1 Sunlight is “white”
Sunlight is a mixture of all visible wavelengths (≈ 380 nm – 750 nm). When it reaches the top of the atmosphere it is essentially white.

### 1.2 Scattering by the atmosphere
When light passes through a medium it can be **absorbed**, **reflected**, or **scattered**. In the clear air above us the dominant scatte

## Bind tools so the model can request tool calls, and ask for typed structured output.

In [14]:
from langchain.tools import tool
from pydantic import BaseModel, Field

@tool
def get_weather(location: str) -> str:
    """gets the weather of a location"""
    return f"weather in {location} is sunny"

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What is the weather in Hyderabad")
for r in response.tool_calls:
    print(r)

class Movie(BaseModel):
    title:str = Field(description="movies title")
    year:int = Field(description="release year")

structured = model.with_structured_output(Movie)
print(structured.invoke("details about Inception"))

{'name': 'get_weather', 'args': {'location': 'Hyderabad'}, 'id': 'chatcmpl-tool-93366c0f18393e6a', 'type': 'tool_call'}
title='Inception' year=2010
